# 05 Wave processing chain

Long swell shows up in a SAR image as regular light and dark stripes, one per crest. The 2D Fourier transform turns the stripes into a bright spot: how far the spot is from the centre gives the wavelength, and which way it sits gives how the crests are lined up. This notebook finds that spot, works out which way the waves are travelling, and checks the answer (report Section 4.4).

Steps:

1. remove the Doppler centroid and set up 45 m square pixels
2. cut the frame into overlapping 12 km tiles and drop the ones with land, ships or rain
3. image spectrum and sub-look cross-spectrum per tile, averaged and rotated north-up
4. whiten the spectrum so the swell peak beats the bright centre, then pick it
5. direction of travel from the sign of the cross-spectrum
6. checks: counting crests along a line, the CMEMS wave partitions, ERA5 wind sea, and the Faraday rotation map

It follows the structure of the López-Dekker and Yuan framework for early Biomass ocean data but was written separately, with the whitening (step 4) and the checks (step 6) added. VV and HH are run separately as a consistency check.

Section 0 runs the chain on simulated swell first, so the method is tested before it's trusted on a real frame.

In [ ]:
# ---- the only cell that needs editing ----
from pathlib import Path

PRODUCT = Path(r"C:\Users\Orlaith.Doyle\BIO_S2_SCS__1S_20260806T....._T008_F199_01_XXXXXX")   # Southern Ocean storm
WAVES = Path(r"C:\Users\Orlaith.Doyle\refs\cmems_mod_glo_wav_anfc_0.083deg_PT3H-i_20260806.nc")  # CMEMS MFWAM, None to skip
ERA5 = Path(r"C:\Users\Orlaith.Doyle\refs\era5_wind_20260806.nc")                                # None to skip
OUT = Path(r"C:\Users\Orlaith.Doyle\biomass_waves")

PIX_M = 45.0            # square pixel size after resampling
TILE_M = 12000.0        # tile size. holds dozens of wavelengths but the sea stays about the same across it
OVERLAP = 0.5
LMIN, LMAX = 120.0, 1000.0   # wavelength band the peak is searched in (m)
AXIS_BINS = 2           # strips this wide along the spectral axes are ignored, processing artefacts collect there

USE_WORLDCOVER = False  # open ocean frames don't need it
SHIP_RATIO = 20.0       # tile dropped if its brightest 45 m cell is this many times the median
MAX_TEXTURE = 0.6       # tile dropped if its normalised std is above this (rain cells, fronts)

CROSS_SIGN = -1         # every tile first pointed exactly opposite to the wave model, the same in every tile,
                        # so it's how the product stores the sub-looks, not the sea. confirm on more scenes!
STEEPNESS = 1 / 15      # ASSUMED wave height / wavelength, only for the labelled estimate at the end.
                        # 1/15 suits steep wind sea, long swell is much gentler (more like 1/50 to 1/200)

In [ ]:
import sys
sys.path.append(str(Path.cwd()))
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.interpolate import RegularGridInterpolator
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
import bm_tools as bm

OUT.mkdir(parents=True, exist_ok=True)

def save(fig, what, tag):
    fig.savefig(OUT / f"{tag}_{what}.png", dpi=200, bbox_inches="tight")
    print("saved", f"{tag}_{what}.png")

## The building blocks

**Sub-looks.** The azimuth spectrum of a tile is split into two halves and each is turned back into an image. They see the same sea a fraction of a second apart, because the two halves of the synthetic aperture point slightly forward and slightly back. The half with positive Doppler is the earlier one.

**Spectra.** Each look is normalised by its own smoothed brightness (so slow brightness changes across the tile don't dominate), windowed and Fourier transformed. The image spectrum is |F(I)|^2. The cross-spectrum is F(I_early) times the conjugate of F(I_late). Waves move a little between the looks, so its imaginary part is positive on the side the waves travel towards and negative on the other side. A single image can't tell north-east from south-west, but this can.

**North-up.** The spectra are on azimuth and range axes. They're interpolated onto a north and east wavenumber grid using the track heading, so they can be compared with the wave model.

**Whitening.** Ocean spectra are always brightest near the centre, so picking the raw maximum mostly finds that background instead of the swell. Dividing by the median at each wavelength (the ring median) means a spot only stands out if it beats everything else at the same wavelength.

**Peak.** The highest whitened point in the wavelength band, refined by a weighted centroid of its neighbours since the spectral bins are coarse (about 2.5 % in wavelength at 300 m).

In [ ]:
NPIX = int(round(TILE_M / PIX_M))
KAX = np.fft.fftshift(np.fft.fftfreq(NPIX, PIX_M)) * 2 * np.pi        # rad/m, same both axes
KG = np.linspace(-2 * np.pi / 90, 2 * np.pi / 90, 2 * NPIX + 1)       # north-up grid, a bit finer
KE, KN = np.meshgrid(KG, KG)                                          # rows = north, cols = east
KMAG = np.hypot(KE, KN)
WIN = np.outer(np.hanning(NPIX), np.hanning(NPIX))


def normalise(I):
    bg = ndimage.gaussian_filter(I, NPIX / 8)
    x = I / bg - 1
    return (x - x.mean()) * WIN


def tile_spectra(I_early, I_late):
    """image spectrum and cross spectrum of one tile, already on the PIX_M grid, fftshifted"""
    Fa = np.fft.fftshift(np.fft.fft2(normalise(I_early)))
    Fb = np.fft.fftshift(np.fft.fft2(normalise(I_late)))
    Fi = np.fft.fftshift(np.fft.fft2(normalise(I_early + I_late)))
    P = np.abs(Fi) ** 2
    X = Fa * np.conj(Fb)
    ax = np.abs(np.arange(NPIX) - NPIX // 2) < AXIS_BINS
    P[ax, :] = np.nan; P[:, ax] = np.nan                  # kill the axis strips
    X[ax, :] = np.nan; X[:, ax] = np.nan
    return P, X


def north_up(A, heading):
    """azimuth/range spectrum -> (north, east) grid. azimuth points along heading, range 90 deg to the
    right of it for a right looking radar (left: -90)"""
    h = np.radians(heading)
    r = np.radians(heading + (90 if LOOK == "right" else -90))
    k_az = KE * np.sin(h) + KN * np.cos(h)
    k_rg = KE * np.sin(r) + KN * np.cos(r)
    f = RegularGridInterpolator((KAX, KAX), A, bounds_error=False, fill_value=np.nan)
    return f(np.column_stack([k_az.ravel(), k_rg.ravel()])).reshape(KE.shape)


def whiten(P):
    edges = np.linspace(0, KMAG.max(), 120)
    idx = np.digitize(KMAG, edges)
    med = np.full(P.shape, np.nan)
    for i in np.unique(idx):
        s = (idx == i) & np.isfinite(P)
        if s.sum() > 5:
            med[idx == i] = np.median(P[s])
    return P / med


def pick_peak(W, Xim, n_top=10):
    band = (KMAG > 2 * np.pi / LMAX) & (KMAG < 2 * np.pi / LMIN) & np.isfinite(W)
    Wb = np.where(band, W, -np.inf)
    i, j = np.unravel_index(np.argmax(Wb), W.shape)
    s = (slice(max(i - 3, 0), i + 4), slice(max(j - 3, 0), j + 4))
    w = np.clip(np.nan_to_num(Wb[s], neginf=0) - 0.5 * W[i, j], 0, None)    # top half of the peak only
    ke, kn = (KE[s] * w).sum() / w.sum(), (KN[s] * w).sum() / w.sum()
    # direction of travel: positive imaginary part on the side the waves go to
    xi = np.nanmean(Xim[s])
    sgn = 1 if xi * CROSS_SIGN_USED > 0 else -1
    ke_t, kn_t = sgn * ke, sgn * kn
    # do the strongest local maxima come in mirrored pairs?
    mx = (Wb == ndimage.maximum_filter(np.nan_to_num(Wb, neginf=0), size=5)) & band
    order = np.argsort(Wb[mx])[::-1][:n_top]
    pts = np.column_stack([KE[mx][order], KN[mx][order]])
    paired = sum(np.min(np.hypot(*(pts + p).T)) < 3 * (KG[1] - KG[0]) for p in pts)
    k = np.hypot(ke, kn)
    return {"wavelength": 2 * np.pi / k,
            "travel_to": np.degrees(np.arctan2(ke_t, kn_t)) % 360,
            "peak_W": W[i, j], "xim": xi, "paired": f"{paired}/{len(pts)}",
            "ke": ke_t, "kn": kn_t}

## 0. Test on simulated swell

Before touching real data: swell of 260 to 420 m travelling a known way, imaged through multiplicative speckle (4 looks, so it's noisy), with the late look taken half a second after the early one so the crests move a little. The tiles go through exactly the same spectra, rotation, whitening and peak picking as the real ones. What comes back should be the right wavelength (within 2 %), the right orientation (within 0.2 deg) and the right direction of travel every time.

Waves running almost exactly along range end up on the azimuth axis of the spectrum, which is masked out (step 4), so those can't be picked. The test directions avoid that on purpose.

This only tests what happens after the sub-look split. The split itself and the sign convention of real products can't be simulated this way, which is why `CROSS_SIGN` is checked against the wave model on real scenes.

In [ ]:
LOOK, CROSS_SIGN_USED = "right", 1          # the simulation uses the textbook sign
rng = np.random.default_rng(3)
yy, xx = np.mgrid[0:NPIX, 0:NPIX] * PIX_M    # azimuth, range in metres


def fake_tile(wl, travel, heading, dt=0.5, looks=4, m=0.3):
    k = 2 * np.pi / wl
    w = np.sqrt(9.81 * k)
    a = np.radians(travel - heading)             # travel direction relative to azimuth
    kaz, krg = k * np.cos(a), k * np.sin(a)
    out = []
    for t in (0, dt):
        I = 1 + m * np.cos(kaz * yy + krg * xx - w * t)
        out.append(I * rng.gamma(looks, 1 / looks, I.shape))
    return out

print(" true wl  true dir | found wl  found dir   error")
for wl, travel, heading in [(260, 37, 350), (300, 200, 190), (360, 120, 10), (420, 300, 168)]:
    Ps, Xs = 0, 0
    for _ in range(40):
        P, X = tile_spectra(*fake_tile(wl, travel, heading))
        Ps, Xs = Ps + P, Xs + X
    Pn = north_up(Ps, heading); Xn = north_up(Xs.imag, heading)
    pk = pick_peak(whiten(Pn), Xn)
    d = (pk["travel_to"] - travel + 180) % 360 - 180
    print(f"{wl:7.0f} {travel:8.0f} | {pk['wavelength']:8.1f} {pk['travel_to']:9.1f}"
          f"   {100 * (pk['wavelength'] - wl) / wl:+.1f} %, {d:+.2f} deg")

## 1. Read the frame and prepare it

Same reading and geocoding as notebook 02 (band order checked, 20 x 20 tie grid). The geocoding gives the ground pixel spacing, the heading for the north-up rotation and the scene centre for the model comparison.

The Doppler centroid is estimated from the phase of the average product of neighbouring azimuth lines and taken out, so the azimuth spectrum is centred on zero and the split in the next step is even.

In [ ]:
slc, order = bm.read_slc(PRODUCT, pols=("HH", "HV", "VH", "VV"))
ann = bm.read_annotation(PRODUCT)
meta = bm.parse_name(PRODUCT.name)
TAG = f"T{meta['track']:03d}_F{meta['frame']:03d}_{meta['start']:%Y%m%dT%H%M}" if meta["start"] else PRODUCT.name[:40]
LOOK, CROSS_SIGN_USED = ann["look"], CROSS_SIGN
shape = slc["VV"].shape

rows_t, cols_t, lat_t, lon_t, inc_t = bm.geocode_grid(ann, shape, n=20, look=LOOK)
d_az, d_rg = bm.pixel_spacing(rows_t, cols_t, lat_t, lon_t)
heading = bm.track_heading(lat_t, lon_t)
lat_c, lon_c = lat_t.mean(), lon_t.mean()
t_c = ann["t0"] + np.timedelta64(int(shape[0] / 2 * ann["dt_az"] * 1e6), "us")
print(TAG, "| bands", order, "|", ann["orbit_pass"], LOOK, "looking")
print("spacing %.2f m az, %.2f m rg | heading %.1f | centre %.3f, %.3f | %s" % (d_az, d_rg, heading, lat_c, lon_c, t_c))

for p in ("VV", "HH"):
    s = slc[p]
    acc = np.nansum(s[1::, ::8][::4] * np.conj(s[:-1:, ::8][::4]))
    fdc = np.angle(acc) / (2 * np.pi * ann["dt_az"])
    slc[p] = s * np.exp(-2j * np.pi * fdc * np.arange(shape[0]) * ann["dt_az"]).astype(np.complex64)[:, None]
    print(f"{p}: doppler centroid {fdc:.1f} Hz removed")

if USE_WORLDCOVER:
    step = 16
    lat_m = bm.grid_interp(rows_t, cols_t, lat_t, np.arange(0, shape[0], step), np.arange(0, shape[1], step))
    lon_m = bm.grid_interp(rows_t, cols_t, lon_t, np.arange(0, shape[0], step), np.arange(0, shape[1], step))
    land = np.repeat(np.repeat(~bm.worldcover_water(lat_m, lon_m), step, 0), step, 1)[:shape[0], :shape[1]]
else:
    land = np.zeros(shape, bool)

## 2 and 3. Tiles, spectra and the average

Each 12 km tile is cut from the complex image, split into sub-looks, turned into intensity and resampled to 45 m square pixels, so a wave is the same size whether it lies along azimuth or range. The Faraday rotation angle (Bickel and Bates) is made on the same tiles, so its spectrum can be checked later.

A tile is dropped if it touches land, if one cell is far brighter than the rest (a ship or an artefact), or if its brightness varies far more than speckle and swell would (rain, a front, a slick). The kept tiles are averaged, which is what makes the peak usable.

In [ ]:
def to_grid(I, cplx=False):
    """native pixels -> NPIX x NPIX at PIX_M. azimuth block-averaged first so it doesn't alias"""
    m = max(1, int(PIX_M // d_az))
    n0 = I.shape[0] // m
    I = I[:n0 * m].reshape(n0, m, -1).mean(1)
    zf = (NPIX / I.shape[0], NPIX / I.shape[1])
    if cplx:
        return ndimage.zoom(I.real, zf, order=1)[:NPIX, :NPIX] + 1j * ndimage.zoom(I.imag, zf, order=1)[:NPIX, :NPIX]
    return ndimage.zoom(I, zf, order=1)[:NPIX, :NPIX]


def sublooks(chip):
    F = np.fft.fft(chip, axis=0)
    f = np.fft.fftfreq(chip.shape[0])[:, None]
    early = np.fft.ifft(np.where(f >= 0, F, 0), axis=0)      # positive doppler, seen first
    late = np.fft.ifft(np.where(f < 0, F, 0), axis=0)
    return np.abs(early) ** 2, np.abs(late) ** 2


nr_t, nc_t = int(TILE_M / d_az), int(TILE_M / d_rg)
r_starts = np.arange(0, shape[0] - nr_t + 1, int(nr_t * (1 - OVERLAP)))
c_starts = np.arange(0, shape[1] - nc_t + 1, int(nc_t * (1 - OVERLAP)))
print(f"tile {nr_t} x {nc_t} px, {len(r_starts) * len(c_starts)} tiles")

acc = {p: {"P": 0, "X": 0, "n": 0} for p in ("VV", "HH")}
accF, why = 0, {"land": 0, "ship": 0, "texture": 0}
ntot = len(r_starts) * len(c_starts)
kept_tiles = []
for it, (r0, c0) in enumerate([(r, c) for r in r_starts for c in c_starts]):
    sl = (slice(r0, r0 + nr_t), slice(c0, c0 + nc_t))
    if land[sl].any():
        why["land"] += 1; continue
    I45 = to_grid(np.abs(slc["VV"][sl]) ** 2)
    if I45.max() / np.median(I45) > SHIP_RATIO:
        why["ship"] += 1; continue
    if np.std(I45 / ndimage.gaussian_filter(I45, NPIX / 8)) > MAX_TEXTURE:
        why["texture"] += 1; continue
    kept_tiles.append((r0, c0))
    for p in ("VV", "HH"):
        Ie, Il = sublooks(slc[p][sl])
        P, X = tile_spectra(to_grid(Ie), to_grid(Il))
        acc[p]["P"] = acc[p]["P"] + P; acc[p]["X"] = acc[p]["X"] + X; acc[p]["n"] += 1
    # faraday rotation map on the same tile
    hh, hv, vh, vv = (slc[q][sl] for q in ("HH", "HV", "VH", "VV"))
    z = (hv - vh + 1j * (hh + vv)) * np.conj(vh - hv + 1j * (hh + vv))
    om = np.angle(to_grid(z, cplx=True)) / 4
    Fo = np.fft.fftshift(np.fft.fft2((om - om.mean()) * WIN))
    accF = accF + np.abs(Fo) ** 2
    if it % 20 == 0:
        print(f"  tile {it + 1}/{ntot}, kept {len(kept_tiles)}")
print("kept", len(kept_tiles), "| dropped:", why)

## 4 and 5. Whitened spectrum, peak and direction of travel

Three panels per channel, like Figure 1 in the report: the smoothed log image spectrum, the whitened spectrum, and the imaginary part of the cross-spectrum (red is the side the waves travel towards). Rings mark wavelengths. Arrows, if the wave model is loaded, are the CMEMS partitions pointing the way they travel.

If VV and HH give the same picture the pattern isn't an artefact of one channel.

In [ ]:
# wave model partitions and ERA5 wind at the scene centre, closest time
parts, wind = [], None
if WAVES and Path(WAVES).exists():
    import xarray as xr
    w = bm.nearest_time(xr.open_dataset(WAVES), t_c)
    for lab, h, tm, dr in [("wind sea", "VHM0_WW", "VTM01_WW", "VMDR_WW"),
                           ("swell 1", "VHM0_SW1", "VTM01_SW1", "VMDR_SW1"),
                           ("swell 2", "VHM0_SW2", "VTM01_SW2", "VMDR_SW2")]:
        if h in w:
            Hs, T, D = (float(bm.interp_ll(w[v], lat_c, lon_c)) for v in (h, tm, dr))
            # CMEMS directions are where the waves come FROM, so +180 to get where they go
            parts.append({"name": lab, "Hs": Hs, "T": T, "wl": 9.81 * T**2 / (2 * np.pi), "to": (D + 180) % 360})
    for p_ in parts:
        print(f"{p_['name']:9s} Hs {p_['Hs']:.1f} m, T {p_['T']:.1f} s, wavelength {p_['wl']:.0f} m, travelling to {p_['to']:.0f}")
if ERA5 and Path(ERA5).exists():
    import xarray as xr
    e = bm.nearest_time(xr.open_dataset(ERA5), t_c)
    u, v = float(bm.interp_ll(e["u10"], lat_c, lon_c)), float(bm.interp_ll(e["v10"], lat_c, lon_c))
    kk = np.logspace(-4, 1, 4000)
    wind = {"U": np.hypot(u, v), "from": (270 - np.degrees(np.arctan2(v, u))) % 360}
    wind["wl_peak"] = 2 * np.pi / kk[np.argmax(bm.elfouhaily(kk, wind["U"]))]
    print("ERA5 wind %.1f m/s from %.0f deg, wind sea peak (Elfouhaily) %.0f m" % (wind["U"], wind["from"], wind["wl_peak"]))

res = {}
fig, axs = plt.subplots(2, 3, figsize=(15, 10), layout="constrained")
lim = 2 * np.pi / 150
for row, p in enumerate(("VV", "HH")):
    Pn = north_up(acc[p]["P"] / acc[p]["n"], heading)
    Xn = north_up((acc[p]["X"] / acc[p]["n"]).imag, heading)
    W = whiten(Pn)
    res[p] = pk = pick_peak(W, Xn)
    print(f"{p}: {pk['wavelength']:.0f} m, travelling to {pk['travel_to']:.0f} deg "
          f"(whitened peak {pk['peak_W']:.1f}, top peaks paired {pk['paired']})")
    Ps = ndimage.gaussian_filter(np.nan_to_num(np.log10(Pn), nan=np.nanmin(np.log10(Pn))), 1)
    xs = np.nanpercentile(np.abs(Xn), 99)
    for a, img, cm, ttl, kw in [(axs[row, 0], Ps, "viridis", "log image spectrum", {}),
                                (axs[row, 1], W, "magma", "whitened", {"vmin": 0, "vmax": np.nanpercentile(W, 99.9)}),
                                (axs[row, 2], Xn * CROSS_SIGN_USED, "RdBu_r", "Im cross-spectrum (red = towards)", {"vmin": -xs, "vmax": xs})]:
        a.pcolormesh(KE, KN, img, cmap=cm, shading="auto", **kw)
        for wl in (200, 300, 500, 800):
            a.add_patch(plt.Circle((0, 0), 2 * np.pi / wl, fill=False, color="w" if cm != "RdBu_r" else "k", lw=0.6, ls=":"))
            a.text(0, 2 * np.pi / wl, f"{wl} m", fontsize=7, color="w" if cm != "RdBu_r" else "k")
        for p_, c in zip(parts, ["c", "lime", "orange"]):
            k = 2 * np.pi / p_["wl"]
            a.annotate("", xy=(k * np.sin(np.radians(p_["to"])), k * np.cos(np.radians(p_["to"]))), xytext=(0, 0),
                       arrowprops=dict(arrowstyle="->", color=c, lw=1.5))
            a.text(k * np.sin(np.radians(p_["to"])), k * np.cos(np.radians(p_["to"])), p_["name"], color=c, fontsize=7)
        a.plot(pk["ke"], pk["kn"], "x", color="r" if cm != "RdBu_r" else "k", ms=9)
        a.set_xlim(-lim, lim); a.set_ylim(-lim, lim); a.set_aspect("equal")
        a.set_title(f"{p}: {ttl}"); a.set_xlabel("k east (rad/m)"); a.set_ylabel("k north (rad/m)")
fig.suptitle(f"{TAG}: {acc['VV']['n']} tiles, north up")
save(fig, "wave_spectra", TAG)
plt.show()

if wind:
    pk = res["VV"]
    print(f"\nobserved {pk['wavelength']:.0f} m is {pk['wavelength'] / wind['wl_peak']:.1f} x the local wind sea peak, "
          f"and travels {abs((pk['travel_to'] - (wind['from'] + 180) + 180) % 360 - 180):.0f} deg away from the wind direction")
    print("-> much longer than the local wind can make means swell from somewhere else" if pk["wavelength"] > 2 * wind["wl_peak"] else "")

## 6. Checks

**Azimuth cutoff.** Water moving towards or away from the radar is put in slightly the wrong place along track, which smears the image in azimuth. Waves shorter than the azimuth cutoff travelling along track can't be imaged. It's estimated by fitting exp(-(pi x / lambda_c)^2) to the azimuth autocorrelation of the image, which comes from the averaged spectrum before rotation.

**Counting crests.** An independent wavelength straight from the image: brightness along a line in the peak direction, averaged over a band of parallel lines to beat the speckle, smoothed, and the distance between maxima taken.

**Faraday.** At high latitude the ionosphere can paint stripes that look like swell. If the Faraday rotation map has its own peak at the same spot, the "swell" is treated as ionospheric.

**Wave height** isn't measured. The estimate below assumes a steepness and is only there as a rough scale, labelled as such everywhere it's used.

In [ ]:
# azimuth cutoff
Pa = np.nan_to_num(acc["VV"]["P"] / acc["VV"]["n"])
prof = Pa.sum(1)
acf = np.real(np.fft.fftshift(np.fft.ifft(np.fft.ifftshift(prof))))
acf = acf / acf.max()
lags = (np.arange(NPIX) - NPIX // 2) * PIX_M
g = np.abs(lags) < 1500
try:
    (lc,), _ = curve_fit(lambda x, l: np.exp(-(np.pi * x / l) ** 2), lags[g], acf[g], p0=[300])
    print("azimuth cutoff about %.0f m" % abs(lc))
except RuntimeError:
    lc = np.nan; print("azimuth cutoff fit didn't converge")

# counting crests along the peak direction, on the tile closest to the frame centre
pk = res["VV"]
r0, c0 = min(kept_tiles, key=lambda t: (t[0] + nr_t / 2 - shape[0] / 2) ** 2 + ((t[1] + nc_t / 2 - shape[1] / 2) * d_rg / d_az) ** 2)
img = to_grid(np.abs(slc["VV"][r0:r0 + nr_t, c0:c0 + nc_t]) ** 2)
img = img / ndimage.gaussian_filter(img, NPIX / 8)
a = np.radians(pk["travel_to"] - heading)
u_az, u_rg = np.cos(a), np.sin(a) * (1 if LOOK == "right" else -1)
s = np.arange(-NPIX / 2 + 10, NPIX / 2 - 10)
lines = []
for off in np.arange(-20, 21):
    rr = NPIX / 2 + s * u_az - off * u_rg
    cc = NPIX / 2 + s * u_rg + off * u_az
    lines.append(ndimage.map_coordinates(img, [rr, cc], order=1, mode="nearest"))
prof = ndimage.gaussian_filter1d(np.mean(lines, 0), pk["wavelength"] / PIX_M / 6)
pks, _ = find_peaks(prof, distance=max(1, int(0.5 * pk["wavelength"] / PIX_M)))
crest = np.mean(np.diff(pks)) * PIX_M if len(pks) > 2 else np.nan
print("crest counting: %d crests, mean spacing %.0f m (spectrum %.0f m)" % (len(pks), crest, pk["wavelength"]))

# faraday spectrum at the swell spot
Fn = whiten(north_up(accF / max(acc["VV"]["n"], 1), heading))
i = np.argmin(np.hypot(KE - pk["ke"], KN - pk["kn"]))
far_peak = Fn.ravel()[i]
print("faraday spectrum at the swell spot: %.1f x its ring median -> %s"
      % (far_peak, "looks IONOSPHERIC, be careful" if far_peak > 3 else "no peak, so it's sea"))

print("\nwave height ESTIMATE (assumed steepness %.3f, not measured): %.0f m"
      % (STEEPNESS, STEEPNESS * pk["wavelength"]))

fig, axs = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
axs[0].plot(lags, acf, label="azimuth ACF")
axs[0].plot(lags, np.exp(-(np.pi * lags / lc) ** 2), "--", label=f"fit, cutoff {abs(lc):.0f} m")
axs[0].set_xlim(-2000, 2000); axs[0].set_xlabel("lag (m)"); axs[0].legend()
axs[1].plot(s * PIX_M / 1000, prof); axs[1].plot(s[pks] * PIX_M / 1000, prof[pks], "rv")
axs[1].set_xlabel("km along travel direction"); axs[1].set_title(f"crests, spacing {crest:.0f} m")
axs[2].pcolormesh(KE, KN, Fn, cmap="magma", shading="auto", vmin=0, vmax=np.nanpercentile(Fn, 99.9))
axs[2].plot(pk["ke"], pk["kn"], "cx", ms=10); axs[2].plot(-pk["ke"], -pk["kn"], "cx", ms=10)
lim = 2 * np.pi / 150
axs[2].set_xlim(-lim, lim); axs[2].set_ylim(-lim, lim); axs[2].set_aspect("equal")
axs[2].set_title("whitened Faraday spectrum (x = swell spot)")
fig.suptitle(TAG)
save(fig, "wave_checks", TAG)
plt.show()

## What doesn't work: wave height from the phase

In principle wave height could come from how fast the surface moves, measured from the phase change between two looks a short time apart, but that needs the two looks to still be coherent. Here it's the complex correlation between the early and late sub-look images on the central tile. On T033 F156 it came out at 0.006, so the phase difference is just noise.

Worth keeping in mind when reading that number: the two looks come from separate halves of the Doppler spectrum, and for speckle those halves are already close to uncorrelated even over a frozen surface. So a near zero value here doesn't on its own prove the sea decorrelated. The real limit is still the P-band aperture time being longer than the ocean's coherence time, but this test can't show that by itself.

In [ ]:
chip = slc["VV"][r0:r0 + nr_t, c0:c0 + nc_t]
F = np.fft.fft(chip, axis=0)
f = np.fft.fftfreq(chip.shape[0])[:, None]
e = np.fft.ifft(np.where(f >= 0, F, 0), axis=0)
l = np.fft.ifft(np.where(f < 0, F, 0), axis=0)
R = np.abs(np.mean(e * np.conj(l))) / np.sqrt(np.mean(np.abs(e) ** 2) * np.mean(np.abs(l) ** 2))
print("early/late sub-look coherence: %.4f" % R)

import json
summary = {"tag": TAG, "tiles": acc["VV"]["n"], "azimuth_cutoff_m": float(abs(lc)), "crest_spacing_m": float(crest),
           "faraday_at_peak": float(far_peak), "VV": {k: v for k, v in res["VV"].items() if k not in ("ke", "kn")},
           "HH": {k: v for k, v in res["HH"].items() if k not in ("ke", "kn")}, "model": parts, "era5": wind,
           "height_ESTIMATE_m": STEEPNESS * res["VV"]["wavelength"], "steepness_ASSUMED": STEEPNESS}
(OUT / f"{TAG}_waves.json").write_text(json.dumps(summary, indent=1, default=float))
print("saved", f"{TAG}_waves.json")